In [1]:
import queue

In [2]:
# Defining datatype
HeuristicMap = dict[str, int]
WeightedGraph = dict[str, dict[str, int]]

## Weighted Romanian (BFS + DFS)

In [3]:
weighted_romanian: WeightedGraph = {
    'Arad': {'Sibiu': 140, 'Zerind': 75, 'Timisoara': 118},
    'Zerind': {'Arad': 75, 'Oradea': 71},
    'Oradea': {'Zerind': 71, 'Sibiu': 151},
    'Sibiu': {'Arad': 140, 'Oradea': 151, 'Fagaras': 99, 'Rimnicu': 80},
    'Timisoara': {'Arad': 118, 'Lugoj': 111},
    'Lugoj': {'Timisoara': 111, 'Mehadia': 70},
    'Mehadia': {'Lugoj': 70, 'Dobreta': 75},
    'Dobreta': {'Mehadia': 75, 'Craiova': 120},
    'Craiova': {'Dobreta': 120, 'Rimnicu': 146, 'Pitesti': 138},
    'Rimnicu': {'Sibiu': 80, 'Craiova': 146, 'Pitesti': 97},
    'Fagaras': {'Sibiu': 99, 'Bucharest': 211},
    'Pitesti': {'Rimnicu': 97, 'Craiova': 138, 'Bucharest': 101},
    'Bucharest': {'Fagaras': 211, 'Pitesti': 101, 'Giurgiu': 90, 'Urziceni': 85},
    'Giurgiu': {'Bucharest': 90},
    'Urziceni': {'Bucharest': 85, 'Vaslui': 142, 'Hirsova': 98},
    'Hirsova': {'Urziceni': 98, 'Eforie': 86},
    'Eforie': {'Hirsova': 86},
    'Vaslui': {'Iasi': 92, 'Urziceni': 142},
    'Iasi': {'Vaslui': 92, 'Neamt': 87},
    'Neamt': {'Iasi': 87}
}

In [4]:
def bfs(graph: WeightedGraph, start_node: str, goal_node: str | None = None) -> None:

    '''
    Performs Breadth-First Search (BFS) on a weighted graph.

    Iterates through the graph layer by layer using a queue. While BFS is
    typically for unweighted graphs to find the shortest path in terms of
    edges, this implementation explores neighbors based on insertion order.

    Args:
        graph (Graph): The adjacency dictionary representing the graph.
        start_node (str): The identifier of the starting node.
        goal_node (str | None): The identifier of the goal node to stop search.
            If None, traverses the entire accessible component.

    Returns:
        None: Prints the traversal path to STDOUT.
    '''

    print(f"--- BFS Traversal starting from {start_node} ---")
    
    visited = []
    q = []
    
    visited.append(start_node)
    q.append(start_node)
    
    while q:
    
        s = q.pop(0)
        print(s, end=" -> ")
        
        if s == goal_node:
            print("(Goal Reached)")
            return

        for neighbour in graph[s].keys():
            if neighbour not in visited:
                visited.append(neighbour)
                q.append(neighbour)
    
    print("End")


def dfs(graph: WeightedGraph, node: str, visited: set[str], goal_node: str | None = None):

    '''
    Performs Depth-First Search (DFS) recursively on a weighted graph.

    Explore as far as possible along each branch before backtracking.

    Args:
        graph (Graph): The adjacency dictionary representing the graph.
        node (str): The current node being visited.
        visited (set[str]): A set tracking visited nodes to prevent cycles.
        goal_node (str | None): The identifier of the goal node.

    Returns:
        bool: True if the goal_node is found, False otherwise.
    '''

    if node not in visited:
    
        print(node, end=" -> ")
        visited.add(node)
        
        if node == goal_node:
            print("(Goal Reached)")
            return True # Signal to stop
        
        for neighbour in graph[node].keys():
            if dfs(graph, neighbour, visited, goal_node):
                return True
    
    return False

In [5]:
bfs(weighted_romanian, 'Arad', 'Bucharest')

--- BFS Traversal starting from Arad ---
Arad -> Sibiu -> Zerind -> Timisoara -> Oradea -> Fagaras -> Rimnicu -> Lugoj -> Bucharest -> (Goal Reached)


In [6]:
dfs_visited = set()
dfs(weighted_romanian, 'Arad', dfs_visited, 'Bucharest')

Arad -> Sibiu -> Oradea -> Zerind -> Fagaras -> Bucharest -> (Goal Reached)


True

## Heuristic A-H Search (A*)

In [7]:
weighted_a2h: WeightedGraph = {
    'A': {'B': 13, 'C': 7, 'F': 5},
    'B': {'A': 13, 'D': 3, 'H': 3},
    'C': {'A': 7, 'D': 5, 'E': 1, 'G': 5},
    'D': {'B': 3, 'C': 5, 'H': 2},
    'E': {'C': 1, 'G': 4},
    'F': {'A': 5, 'G': 6},
    'G': {'C': 5, 'E': 4, 'F': 6},
    'H': {'B': 3, 'D': 2}
}

heuristics_a2h: HeuristicMap = {
    'A': 14,
    'B': 3,
    'C': 7,
    'D': 2,
    'E': 6,
    'F': 10,
    'G': 11,
    'H': 0
}

In [8]:
def a_star(graph: WeightedGraph, source: str, destination: str, heuristic_map: HeuristicMap) -> tuple[int, int, list[str] | None]:

    '''
    Finds the optimal path from source to destination using the A* algorithm.

    Uses a priority queue to explore paths based on the cost function:

    $$$
    f(n) = g(n) + h(n)
    $$$
    
    where g(n) is the cost from the start node and
    h(n) is the estimated cost to the goal.

    Args:
        graph (Graph): The adjacency dictionary with edge weights.
        source (str): The starting node identifier.
        destination (str): The goal node identifier.
        heuristic_map (HeuristicMap): A dictionary mapping nodes to their
            heuristic values (estimated distance to goal).

    Returns:
        tuple[int, int, list[str] | None]: A tuple containing:
            - Final f-score (int) or None if no path found.
            - Total path cost (int) or None.
            - List of nodes representing the path (List[str]) or None.
    '''

    print(f"--- A* Search from {source} to {destination} ---")
    
    pq = queue.PriorityQueue()
    pq.put((0 + heuristic_map[source], 0, source, [source]))
    
    visited_costs = {}
    visited_costs[source] = 0
    
    while not pq.empty():

        (f_score, current_cost, vertex, path) = pq.get()
        
        if vertex == destination:
            return f_score, current_cost, path
        
        for next_node, edge_weight in graph[vertex].items():
            new_cost = current_cost + edge_weight
            
            if next_node not in visited_costs or new_cost < visited_costs[next_node]:
                visited_costs[next_node] = new_cost
                priority = new_cost + heuristic_map[next_node]
                pq.put((priority, new_cost, next_node, path + [next_node]))
                
    return None, None, None

In [9]:
starting_city = 'A'
goal_city = 'H'

f_val, cost, path = a_star(weighted_a2h, starting_city, goal_city, heuristics_a2h)

if path:
    print(f"Optimal Path found: {' -> '.join(path)}")
    print(f"Total Cost: {cost}")
    print(f"Final Heuristic Value (at Goal): {heuristics_a2h[goal_city]}")

else:
    print("No path found.")

--- A* Search from A to H ---
Optimal Path found: A -> C -> D -> H
Total Cost: 14
Final Heuristic Value (at Goal): 0
